<a href="https://colab.research.google.com/github/nashranoor98/hospital-readmission-prediction/blob/main/CaseStudy1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hospital Readmission Prediction
Predicting 30-day hospital readmission using Logistic Regression with L2 regularization.

1. Importing and Studying Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = pd.read_csv("data/diabetic_data.csv")
data = data.replace("?", np.nan)

print(data.shape)
print(data.head())


In [ ]:
print(data.info())
print(data.describe())
print(data.isnull().sum().sort_values(ascending=False).head(10))
print("Duplicate rows:", data.duplicated().sum())


2. EDA and Visualisation

In [ ]:
print(data["readmitted"].value_counts())

plt.figure(figsize=(7,5))
sns.countplot(data=data, x="readmitted", order=["NO", ">30", "<30"])
plt.title("Readmission Distribution")
plt.xlabel("Readmission Category")
plt.ylabel("Number of Patients")
plt.show()


In [ ]:
numeric_cols = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient"
]

plt.figure(figsize=(10,8))
for i, col in enumerate(numeric_cols):
    plt.subplot(3,3,i+1)
    sns.histplot(data[col].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.show()


3. Splitting and Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data["target_30day"] = (data["readmitted"] == "<30").astype(int)

drop_cols = ["encounter_id", "patient_nbr", "readmitted", "target_30day"]
X = data.drop(columns=drop_cols)
y = data["target_30day"]

high_missing = X.columns[X.isna().mean() > 0.90]
X = X.drop(columns=high_missing)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric = X_train.select_dtypes(include=np.number).columns
categorical = X_train.select_dtypes(exclude=np.number).columns

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical)
])

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])
print("Features retained:", X.shape[1])


4. Training and Evaluating Model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(
        penalty="l2", C=1.0, max_iter=1000, solver="liblinear"
    ))
])

model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)


In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay

print("ROC_AUC_SCORE:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("Classification Report:")
print(classification_report(y_test, y_pred))

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve")
plt.show()


5. False Negative vs False Positive

False negatives can be clinically costly because a high-risk patient may be missed. False positives may lead to additional follow-up or resource use. Therefore, threshold selection should consider the clinical cost of missed readmissions.